# 🌳 10장 · 스무고개와 집단지성: 결정나무와 앙상블 (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 10장의 **04절 예제 확인**, **05. 파이썬 실습**, **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 10-1, 10-3, 10-6, 10-7, 10-8, 10-9 |
| ☆ 선택 | 자습이나 심화 | 10-2, 10-4, 10-5 |

**데이터**
- 은행 전화 마케팅: UCI "Bank Marketing"(bank-full) 45,211건. GitHub 저장소 `khatrideepti/Bank-Marketing-Data-Analysis`에서 바로 읽습니다.
- 미니 프로젝트: UCI "Adult"(1994년 미국 인구조사) 48,842명. GitHub 저장소 `jbrownlee/Datasets`에서 바로 읽습니다.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다. 나무 300그루의 숲을 여러 번 키우므로, 05절 전체가 코랩에서 몇 분 걸립니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 사이킷런, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib

## 04 · 손 계산을 파이썬으로 확인하기

책 04절의 예제를 손으로 푼 뒤, 같은 계산을 파이썬으로 확인합니다.

### [예제 10-1] 고객 10명을 나누는 두 질문

고객 10명 가운데 4명이 가입했습니다. 질문 A는 이들을 4명(가입 3명)과 6명(가입 1명)으로, 질문 B는 5명(가입 2명)씩 둘로 나눕니다. 각 질문의 값(지니 불순도 감소)을 구하시오.

In [ ]:
def gini(p):
    return 1 - p**2 - (1 - p)**2
before = gini(4 / 10)                                  # 10명 가운데 4명 가입
after_A = 4 / 10 * gini(3 / 4) + 6 / 10 * gini(1 / 6)  # 질문 A로 나눈 뒤
after_B = 5 / 10 * gini(2 / 5) + 5 / 10 * gini(2 / 5)  # 질문 B로 나눈 뒤
print(round(before, 3), round(after_A, 3), round(after_B, 3))
print("질문 A의 값:", round(before - after_A, 3), " 질문 B의 값:", round(before - after_B, 3))

### 4.2 다수결의 수학

**[예제 10-2] 세 사람의 다수결**

옳게 판단할 확률이 각각 0.6인 세 사람이 서로 상관없이 판단해 다수결로 정합니다. 다수결이 옳을 확률을 구하시오.

사람 수를 늘려 가며 (식 10-3)을 계산합니다.

In [ ]:
from math import comb
def vote(n, p):                                        # n명이 저마다 p로 맞힐 때 다수결이 맞을 확률
    return sum(comb(n, k) * p**k * (1 - p)**(n - k) for k in range(n // 2 + 1, n + 1))
for n in [1, 3, 11, 101, 501]:
    print(f"{n:3d}명: p=0.6 → {vote(n, 0.6):.3f},  p=0.55 → {vote(n, 0.55):.3f}")

### 4.3 닮은 나무의 평균

나무 B그루를 평균한 예측의 분산 ρσ² + (1 − ρ)σ²/B 에 σ² = 1을 넣어 봅니다(식 10-4).

In [ ]:
for rho in [0, 0.3, 1]:                                # 나무끼리의 상관
    print(f"상관 {rho}:", [round(rho + (1 - rho) / B, 3) for B in [1, 10, 300]])

### [예제 10-3] 고객 넷으로 해 보는 부스팅

고객 넷 가운데 첫째만 가입했습니다(y = 1, 0, 0, 0). 처음에는 모두에게 가입 비율 0.25를 줍니다. 나무가 잔차를 정확히 맞힌다고 하고, 학습률 0.5로 나무 두 그루를 더했을 때 첫째 고객의 예측을 구하시오.

(학습률 0.5로 나무 세 그루까지 더해 봅니다.)

In [ ]:
import numpy as np
y4 = np.array([1, 0, 0, 0])                            # 고객 넷 가운데 첫째만 가입
F = np.full(4, y4.mean())                              # 처음에는 모두 0.25
for m in range(3):
    r = y4 - F                                         # 잔차 = 실제 - 예측
    F = F + 0.5 * r                                    # 잔차를 맞힌 나무를 학습률 0.5만큼 더한다
    print(f"나무 {m + 1}그루 뒤 예측:", F.round(3), " 첫째 고객의 잔차:", round(y4[0] - F[0], 3))

## 05. 파이썬 실습: 은행 기록으로 나무와 숲 키우기

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 은행 기록은 깃허브에 올라 있는 UCI 원본 사본에서 바로 읽습니다. 결정나무와 숲은 사이킷런으로 만듭니다. 나무 300그루를 키우는 실습이 여러 개라, 노트북 전체를 실행하면 몇 분이 걸립니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [파이썬 실습 10-1 ★] 데이터 불러오기와 연도 되살리기

은행 기록을 읽고 가입 비율을 구하시오. 달이 거꾸로 가는 곳에서 해가 바뀐다고 보고 연도를 붙인 뒤, 연도별 기록 수와 가입률을 구하시오.

① 데이터를 읽고 가입 여부를 0과 1로 바꿉니다. ② 연도를 되살려 연도별로 묶습니다. ③ 앞 75%와 뒤 25%가 갈리는 날짜를 확인합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

URL = ("https://raw.githubusercontent.com/khatrideepti/"
       "Bank-Marketing-Data-Analysis/master/bank-full.csv")
bank = pd.read_csv(URL, sep=";")
y = (bank["y"] == "yes").astype(int)                  # 가입 1, 안 함 0
print(bank.shape, " 가입:", y.sum(), "명, 비율", round(y.mean(), 3))
print("빈칸:", bank.isna().sum().sum(), " 'unknown' 연락 수단:",
      (bank["contact"] == "unknown").sum())

MONTHS = ["jan", "feb", "mar", "apr", "may", "jun",
          "jul", "aug", "sep", "oct", "nov", "dec"]
m = bank["month"].map({k: i for i, k in enumerate(MONTHS)}).values
bank["year"] = 2008 + np.r_[0, (m[1:] < m[:-1]).cumsum()]  # 달이 거꾸로 가면 새해
by_year = bank.groupby("year").agg(
    기록=("y", "size"), 가입률=("y", lambda s: (s == "yes").mean()),
    연락수단_모름=("contact", lambda s: (s == "unknown").mean()))
print(by_year.round(3))

cut = int(len(bank) * 0.75)                            # 앞 75%로 배우고 뒤 25%로 시험
for name, row in [("훈련 끝", bank.iloc[cut - 1]), ("시험 시작", bank.iloc[cut])]:
    print(name, row["year"], row["month"])

by_year["가입률"].plot.bar(color="#E04B3C", figsize=(5, 3))
plt.xlabel("연도")
plt.ylabel("가입률")
plt.xticks(rotation=0)
plt.show()

칸 구분 기호가 세미콜론이라 `sep=";"`을 주었습니다. 빈칸은 0개이지만 연락 수단 13,020건이 "unknown"입니다. 연도를 되살리니 가입률이 해마다 세 배씩 뛰었습니다. 앞 75%와 뒤 25%는 둘 다 2009년 4월에서 갈립니다. 이 경계가 이 장의 훈련 기간과 시험 기간입니다.

> **[코드 읽기] `np.r_[0, (m[1:] < m[:-1]).cumsum()]`**
>
> `m[1:] < m[:-1]`은 "다음 행의 달이 이번 행의 달보다 작은가"를 행마다 True, False로 냅니다. 12월 다음에 1월이 오는 곳만 True입니다. `cumsum( )`은 True를 1로 세어 차례로 더하므로, 해가 바뀔 때마다 1씩 커지는 번호가 됩니다. `np.r_[0, ...]`는 맨 앞 행에 0을 붙여 길이를 맞춥니다.

#### [파이썬 실습 10-2 ☆] 첫 질문 고르기: 지니 불순도 손 계산

지니 불순도를 함수로 만들고, 첫 질문 후보 네 개가 은행 기록 전체의 불순도를 얼마나 줄이는지 견주시오. 가장 좋은 질문 아래로 한 단계 더 내려가 보시오.

In [ ]:
def gini(p):                                           # 가입 비율 p인 무리의 지니 불순도
    return 1 - p**2 - (1 - p)**2

root = gini(y.mean())
print("나누기 전 지니:", round(root, 4))
questions = {"지난 캠페인 성공?": bank["poutcome"] == "success",
             "나이 60세 초과?": bank["age"] > 60,
             "주택대출 있음?": bank["housing"] == "yes",
             "3월에 연락?": bank["month"] == "mar"}
for q, mask in questions.items():
    a, b = y[mask], y[~mask]                           # 예 / 아니요 두 무리
    after = (len(a) * gini(a.mean()) + len(b) * gini(b.mean())) / len(y)
    print(f"{q:10s} 예 {len(a):5d}명 {a.mean():.3f} / 아니요 {len(b):5d}명 {b.mean():.3f}"
          f"  지니 감소 {root - after:.4f}")

succ = questions["지난 캠페인 성공?"]                   # 둘째 질문까지 내려가 보기
for name, grp, q2, label in [("성공한 고객", succ, bank["housing"] == "yes", "주택대출"),
                             ("나머지 고객", ~succ, bank["age"] > 60, "60세 초과")]:
    for ans in [True, False]:
        g = y[grp & (q2 == ans)]
        print(f"{name} → {label} {'예' if ans else '아니요'}: {len(g):5d}명, 가입률 {g.mean():.3f}")

나누기 전 지니는 0.2066입니다. "지난 캠페인 성공?"이 0.0194를 줄여 다른 후보의 네 배쯤 됩니다. 예라고 답한 1,511명의 가입률이 64.7%로 크게 갈리기 때문입니다. 아래 네 줄이 [그림 @tree_whole]의 잎 네 개입니다.

#### [파이썬 실습 10-3 ★] 시간 순서로 나누고 깊이 3의 나무 키우기

범주 변수를 0과 1의 열로 바꾸고(통화 시간은 뺍니다), 앞 75%로 배우고 뒤 25%로 시험하시오. 기준선 정확도를 구하고, 깊이 3의 나무를 키워 첫 질문과 시험 성적을 확인하시오.

① `get_dummies( )`로 범주를 열로 바꿉니다. ② `iloc`으로 시간 순서대로 자릅니다. ③ 나무를 키우고 그림으로 봅니다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import roc_auc_score

X = pd.get_dummies(bank.drop(columns=["y", "year", "duration"]),
                   dtype=int)                          # 통화 시간은 뺀다(3.7)
Xtr, Xte = X.iloc[:cut], X.iloc[cut:]                  # 시간 순서로 나누기
ytr, yte = y.iloc[:cut], y.iloc[cut:]
print("특징", X.shape[1], "개 / 훈련", len(Xtr), "/ 시험", len(Xte))
print("가입률: 훈련", round(ytr.mean(), 3), " 시험", round(yte.mean(), 3),
      " 시험 기간 가입:", yte.sum(), "건")
print("모두 '안 함'으로 찍은 정확도:", round(1 - yte.mean(), 3))

def auc(model, Xs, ys):                                # 확률로 줄 세우는 능력
    return roc_auc_score(ys, model.predict_proba(Xs)[:, 1])

t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)
print(export_text(t3, feature_names=list(X.columns), max_depth=1))
print("깊이 3 나무: 정확도", round(t3.score(Xte, yte), 3),
      " AUC", round(auc(t3, Xte, yte), 3))
for mo in ["mar", "oct"]:                              # 훈련 기간의 3월과 10월
    k = (bank["month"].iloc[:cut] == mo).values
    print(mo, k.sum(), "건, 연도", sorted(set(bank["year"].iloc[:cut][k])),
          " 가입률", round(ytr[k].mean(), 3))

from sklearn.tree import plot_tree
plt.figure(figsize=(10, 4))
plot_tree(t3, max_depth=2, feature_names=list(X.columns), class_names=["안 함", "가입"],
          filled=True, impurity=False, proportion=True, fontsize=9)
plt.show()

통화 시간을 뺀 변수 15개가 0과 1의 열로 펼쳐져 특징이 50개가 되었습니다. 시험 기간의 가입률(27.8%)이 훈련 기간(6.3%)보다 훨씬 높아, 기준선 정확도는 72.2%입니다. 나무의 첫 질문은 `month_mar`, 곧 "3월에 연락했나?"입니다. 그림의 value는 [안 함, 가입]의 비율이고, 오른쪽 가지의 3월 고객은 48.4%가 가입했습니다. 정확도는 72.6%로 찍기와 거의 같고, AUC는 0.557입니다. 훈련 기간의 3월 기록 258건은 모두 2009년의 것이고, 10월 기록 80건은 모두 2008년의 것입니다.

> **[코드 읽기] `get_dummies( )`와 `iloc`**
>
> `pd.get_dummies( )`는 "job" 같은 범주 열을 `job_admin.`, `job_student`처럼 범주마다 0과 1의 열로 펼칩니다. 나무는 "이 열이 0.5보다 큰가?", 곧 "이 범주인가?"를 묻게 됩니다. `X.iloc[:cut]`은 0번부터 cut − 1번 행까지, `X.iloc[cut:]`은 그 뒤를 고릅니다. 행이 날짜순이라 시간 순서대로 자른 것이 됩니다.

#### [파이썬 실습 10-4 ☆] 끝까지 자란 나무와 깊이 곡선

깊이를 정하지 않고 나무를 끝까지 키워 훈련과 시험 성적을 견주시오. 깊이를 바꾸며 훈련 AUC와 시험 AUC를 그리시오.

In [ ]:
full = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
print("끝까지 자란 나무: 깊이", full.get_depth(), " 잎", full.get_n_leaves())
print("훈련 정확도", round(full.score(Xtr, ytr), 3),
      "/ 시험 정확도", round(full.score(Xte, yte), 3),
      "/ 시험 AUC", round(auc(full, Xte, yte), 3))

rows = []
for depth in [1, 2, 3, 4, 6, 8, 10, 15, 20]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xtr, ytr)
    rows.append([depth, auc(t, Xtr, ytr), auc(t, Xte, yte)])
curve = pd.DataFrame(rows, columns=["깊이", "훈련 AUC", "시험 AUC"])
print(curve.round(3).to_string(index=False))

curve.plot(x="깊이", y=["훈련 AUC", "시험 AUC"], marker="o",
           color=["#3F7CC0", "#E04B3C"], figsize=(6, 3))
plt.ylabel("AUC")
plt.show()

끝까지 자란 나무는 깊이 43, 잎 3,459개로 훈련 기록을 100% 맞혔지만, 시험 정확도는 67.2%로 기준선보다 낮습니다. 깊이 곡선에서 훈련 AUC는 깊이 20에서 0.932까지 오르는데, 시험 AUC는 깊이 8의 0.623이 가장 높습니다. 1.3의 숫자들이 여기서 나왔습니다. 그렇다고 이 곡선을 보고 깊이를 고르면 시험을 미리 본 셈이 됩니다.

#### [파이썬 실습 10-5 ☆] 교차검증으로 잎의 크기 고르기

훈련 기간의 기록만으로 5겹 교차검증을 해서, 잎마다 남길 최소 고객 수를 고르시오.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=0)  # 훈련 기간을 다섯 조각으로
for n in [10, 50, 100, 200, 400]:
    s = cross_val_score(DecisionTreeClassifier(min_samples_leaf=n, random_state=0),
                        Xtr, ytr, cv=cv, scoring="roc_auc")
    print(f"잎마다 최소 {n:3d}명: 검증 AUC {s.mean():.3f}")

교차검증은 훈련 기록을 다섯 조각으로 나누어, 넷으로 배우고 남은 하나로 검증하기를 다섯 번 되풀이하는 방법입니다. 시험 기록은 한 번도 보지 않습니다. 100명일 때 검증 AUC가 0.650으로 가장 높습니다. [파이썬 실습 10-7]의 가지친 나무가 쓴 100명과 같습니다. 다만 이 교차검증은 훈련 기간 안에서 기록을 섞어 나누었습니다. 시간 순서까지 지키려면 훈련 기간의 마지막 몇 달을 검증용으로 떼어 두면 됩니다.

#### [파이썬 실습 10-6 ★] 다르게 틀려야 모은 보람이 있다

똑같은 나무 300그루, 배깅, 랜덤 포레스트를 키워 나무 한 그루와 숲의 성적, 첫 질문의 가짓수를 견주시오. 랜덤 포레스트에서 평균하는 나무 수를 늘려 가며 AUC를 구하시오.

① 세 숲을 같은 함수로 키웁니다. ② 숲 속 나무 30그루를 하나씩 시험합니다. ③ 앞의 k그루만 평균해 봅니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def forest(**kw):
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                  random_state=0, n_jobs=-1, **kw).fit(Xtr, ytr)

kinds = {"똑같은 나무": forest(max_features=None, bootstrap=False),
         "배깅": forest(max_features=None),
         "랜덤 포레스트": forest(max_features="sqrt")}
rows = []
for name, f in kinds.items():
    one = [roc_auc_score(yte, t.predict_proba(Xte.values)[:, 1])
           for t in f.estimators_[:30]]                # 숲 속 나무 30그루를 하나씩
    first = {X.columns[t.tree_.feature[0]] for t in f.estimators_}
    rows.append([name, np.mean(one), auc(f, Xte, yte), len(first)])
crowd = pd.DataFrame(rows, columns=["숲", "나무 하나(평균)", "숲 전체", "첫 질문 종류"])
print(crowd.round(3).to_string(index=False))

rf = kinds["랜덤 포레스트"]
probs = np.array([t.predict_proba(Xte.values)[:, 1] for t in rf.estimators_])
for k in [1, 3, 10, 30, 100, 300]:                     # 앞의 k그루만 평균하면
    print(f"나무 {k:3d}그루 평균: 시험 AUC {roc_auc_score(yte, probs[:k].mean(0)):.3f}")

crowd.set_index("숲")[["나무 하나(평균)", "숲 전체"]].plot.barh(
    color=["#8EC5E8", "#E04B3C"], figsize=(6, 3))
plt.xlim(0.5, 0.75)
plt.xlabel("시험 AUC")
plt.legend(loc="lower right")
plt.show()

`bootstrap=False`와 `max_features=None`이면 모든 나무가 같은 기록, 같은 변수로 자라 거의 똑같아집니다. 첫 질문이 1가지뿐이고 숲(0.587)이 나무 한 그루(0.584)와 같습니다. 배깅은 첫 질문이 3가지로 갈려 0.671, 랜덤 포레스트는 27가지로 갈려 0.708입니다. 나무를 1그루에서 300그루로 늘리면 0.615에서 0.708로 오르지만, 100그루(0.703) 뒤로는 거의 그대로입니다.

> **[코드 읽기] 설정 묶음 `kw`와 `estimators_`**
>
> `def forest(**kw)`는 이름을 붙여 넘긴 설정(`max_features=None` 등)을 모두 `kw`라는 사전으로 받아, 다시 `**kw`로 모델에 그대로 넘깁니다. 숲을 세 번 만드는 코드를 한 번만 쓰려는 것입니다. 다 키운 숲의 `estimators_`에는 나무 300그루가 목록으로 들어 있어 하나씩 꺼내 볼 수 있습니다.

#### [파이썬 실습 10-7 ★] 부스팅과 로지스틱 회귀 견주기

그레이디언트 부스팅을 학습률 0.05와 1.0으로 키워 나무 수에 따른 시험 AUC를 구하시오. 잎마다 100명을 남긴 가지친 나무와 9장의 로지스틱 회귀까지 다섯 모델의 AUC와, 상위 10%와 30%에게 전화할 때 만나는 가입자의 비율을 구하시오.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def boost(rate, Xa=Xtr, ya=ytr):
    return GradientBoostingClassifier(n_estimators=300, learning_rate=rate,
                                      max_depth=3, subsample=0.8,
                                      random_state=0).fit(Xa, ya)

gb, fast = boost(0.05), boost(1.0)
slow_auc = [roc_auc_score(yte, p[:, 1]) for p in gb.staged_predict_proba(Xte)]
fast_auc = [roc_auc_score(yte, p[:, 1]) for p in fast.staged_predict_proba(Xte)]
for k in [1, 10, 50, 100, 300]:
    print(f"나무 {k:3d}그루: 학습률 0.05 → {slow_auc[k - 1]:.3f},"
          f" 학습률 1.0 → {fast_auc[k - 1]:.3f}")
print("학습률 0.05가 가장 좋았던 곳:", int(np.argmax(slow_auc)) + 1, "그루",
      round(max(slow_auc), 3))

pruned = DecisionTreeClassifier(min_samples_leaf=100, random_state=0)
pruned.fit(Xtr, ytr)                                   # 가지친 나무: 잎마다 고객 100명 이상
print("가지친 나무: 잎", pruned.get_n_leaves())
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
lr.fit(Xtr, ytr)
board = {"깊이 3 나무": t3, "가지친 나무": pruned, "로지스틱 회귀(9장)": lr,
         "랜덤 포레스트": rf, "그레이디언트 부스팅": gb}
rows = []
for name, model in board.items():
    p = model.predict_proba(Xte)[:, 1]
    top = np.argsort(-p)                               # 확률 높은 순서
    found = yte.values[top].cumsum() / yte.sum()
    rows.append([name, auc(model, Xte, yte), found[len(p) // 10 - 1],
                 found[len(p) * 3 // 10 - 1]])
score = pd.DataFrame(rows, columns=["모델", "시험 AUC", "상위 10%가 만난 가입자",
                                    "상위 30%가 만난 가입자"])
print(score.round(3).to_string(index=False))

plt.figure(figsize=(6, 3))
plt.plot(range(1, 301), slow_auc, color="#E04B3C", label="학습률 0.05")
plt.plot(range(1, 301), fast_auc, color="#3F7CC0", label="학습률 1.0")
plt.xlabel("더한 나무의 수")
plt.ylabel("시험 AUC")
plt.legend()
plt.show()

`staged_predict_proba( )`는 나무를 1그루, 2그루, … 더할 때마다의 확률을 차례로 돌려주어, 한 번 키운 모델로 곡선을 그릴 수 있습니다. 학습률 0.05는 100그루에서 0.728이다가 300그루에서 0.692로 내려갑니다. 가지친 나무는 잎이 251개이고 시험 AUC는 0.639입니다. 성적표에서는 로지스틱 회귀가 AUC 0.741, 상위 30%에서 55.0%로 가장 좋았습니다. `np.argsort(-p)`는 확률이 높은 순서의 번호이고, `cumsum( )`으로 위에서부터 만난 가입자를 차례로 셉니다.

#### [파이썬 실습 10-8 ★] 통화 시간과 두 가지 중요도

통화 시간을 넣은 부스팅의 시험 AUC와 통화 시간의 중요도를 구하고, 통화 시간별 가입률을 구하시오. 통화 시간을 뺀 부스팅에서 불순도 중요도와 순열 중요도를 견주고, 3월 기록이 어느 해의 것인지 확인하시오.

In [ ]:
from sklearn.inspection import permutation_importance

Xd = pd.get_dummies(bank.drop(columns=["y", "year"]), dtype=int)  # 통화 시간 포함
gbd = boost(0.05, Xd.iloc[:cut], ytr)
print("통화 시간을 넣으면: 시험 AUC", round(auc(gbd, Xd.iloc[cut:], yte), 3),
      " 통화 시간의 중요도", round(gbd.feature_importances_[Xd.columns.get_loc("duration")], 3))
mins = pd.cut(bank["duration"] / 60, [0, 1, 3, 5, 10, 100], right=False)
print(y.groupby(mins, observed=True).mean().round(3).to_string())

imp = pd.Series(gb.feature_importances_, X.columns)    # 통화 시간 없는 부스팅
print("불순도 중요도 상위 5\n", imp.sort_values(ascending=False).head(5).round(3))
pi = permutation_importance(gb, Xte, yte, scoring="roc_auc", n_repeats=5,
                            random_state=0, n_jobs=-1)
perm = pd.Series(pi.importances_mean, X.columns)
print("섞었을 때 떨어지는 AUC 상위 5\n", perm.sort_values(ascending=False).head(5).round(4))
mar = bank[bank["month"] == "mar"]
print("3월 기록:", len(mar), "건, 연도", sorted(mar["year"].unique().tolist()),
      " 가입률", round((mar["y"] == "yes").mean(), 3))

perm.sort_values().tail(8).plot.barh(color="#E04B3C", figsize=(5, 3.5))
plt.xlabel("섞었을 때 떨어진 시험 AUC")
plt.show()

통화 시간을 넣으면 AUC가 0.811로 오르고, 통화 시간 하나의 중요도가 0.636입니다. 1분이 안 된 통화의 가입률은 0.2%, 10분 넘는 통화는 48.3%입니다. 통화 시간을 뺀 모델에서는 불순도 중요도 1, 2위가 날짜(일)와 3월이고, 순열 중요도 1위는 지난 캠페인 성공입니다. 3월 기록 477건은 모두 2009년과 2010년의 것입니다.

#### [파이썬 실습 10-9 ★] 나누는 방법을 바꾸면

가지친 나무, 부스팅, 로지스틱 회귀를 무작위 분할로 시험하고, 시간 분할의 자르는 곳을 80%로 옮겨 부스팅과 로지스틱 회귀를 견주시오.

In [ ]:
from sklearn.model_selection import train_test_split
Rtr, Rte, rtr, rte = train_test_split(X, y, test_size=0.25, stratify=y,
                                      random_state=42)   # 세 해를 섞어 무작위로
mixed = DecisionTreeClassifier(min_samples_leaf=100, random_state=0).fit(Rtr, rtr)
print("무작위로 나누면: 가지친 나무", round(auc(mixed, Rte, rte), 3))
lrr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Rtr, rtr)
print("무작위로 나누면: 부스팅", round(auc(boost(0.05, Rtr, rtr), Rte, rte), 3),
      " 로지스틱", round(auc(lrr, Rte, rte), 3))
c8 = int(len(X) * 0.8)                                 # 자르는 곳을 80%로 옮기면
lr8 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
lr8.fit(X.iloc[:c8], y.iloc[:c8])
print("80%에서 자르면: 부스팅", round(auc(boost(0.05, X.iloc[:c8], y.iloc[:c8]),
                                         X.iloc[c8:], y.iloc[c8:]), 3),
      " 로지스틱", round(auc(lr8, X.iloc[c8:], y.iloc[c8:]), 3))
print("80% 지점:", bank["year"].iloc[c8], bank["month"].iloc[c8])

무작위로 나누면 가지친 나무가 0.771로, 시간 분할(0.639)보다 0.132 높게 나옵니다. 3.2에서 말한 "섞어서 나누면 쉬워진 시험"의 크기입니다. 부스팅도 0.801로, 시간 분할(0.692)보다 0.109 높습니다. 로지스틱 회귀(0.773)보다도 높아서, 무작위 분할만 보았다면 부스팅을 골랐을 것입니다. 자르는 곳을 2009년 5월(80%)로 옮기면 부스팅 0.692, 로지스틱 회귀 0.677로 순서가 바뀝니다. 0.01~0.05의 차이로 모델의 순위를 정하기 전에, 여러 시점에서 되풀이해 보아야 하는 까닭입니다.

> **핵심 3줄**
>
> 사이킷런의 나무, 숲, 부스팅은 같은 `fit`과 `predict_proba`로 쓰고, 성적은 시험 기간의 AUC로 견줍니다.
>
> 중요도는 모델이 무엇을 보았는지 알려 주지만, 불순도 중요도와 순열 중요도가 다를 수 있으니 둘을 함께 봅니다.
>
> 나무 300그루를 모으자 AUC가 0.58쯤에서 0.708로 올랐고, 무작위로 나누면 가지친 나무가 0.771로 시간 분할(0.639)보다 0.132 높게 나왔습니다.

## [미니 프로젝트 10] 연 소득 5만 달러를 넘을지 나무로 맞힐 수 있을까

**질문.** 나이, 학력, 직업, 결혼 여부만 보고 그 사람의 연 소득이 5만 달러를 넘는지 맞힐 수 있을까요? 맞힌다면 모델은 무엇을 보고 맞혔을까요?

**데이터.** 1994년 미국 인구조사에서 뽑은 성인 48,842명의 기록입니다(UCI "Adult"). 한 행은 한 사람이고, 열은 15개입니다. 나이, 일하는 곳의 종류, 학력, 결혼 여부, 직업, 가족 관계, 인종, 성별, 자본 소득, 주당 일하는 시간, 출신 나라, 그리고 소득이 5만 달러를 넘는지가 들어 있습니다. fnlwgt라는 열은 "이 한 사람이 미국 인구 몇 명을 대표하는가"를 나타내는 조사 가중치입니다. 한국 데이터로는 통계청 가계금융복지조사 마이크로데이터로 같은 질문을 할 수 있습니다. 자료를 내려받을 수 있으면 같은 단계로 바꾸어 해 보십시오.


**1단계: 데이터 신원 확인**([파이썬 실습 10-1] 응용). 데이터를 읽고 크기, 똑같은 행, "?"로 적힌 칸, 소득 5만 달러 초과 비율을 구하시오. 성별로도 나누어 보시오.

In [ ]:
COLS = ["age", "workclass", "fnlwgt", "education", "education_num", "marital",
        "occupation", "relationship", "race", "sex", "capital_gain",
        "capital_loss", "hours", "country", "income"]
ADULT = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"
adult = pd.read_csv(ADULT, header=None, names=COLS)
print(adult.shape, " 똑같은 행:", adult.duplicated().sum())
print("'?'로 적힌 칸:", (adult == "?").sum()[lambda s: s > 0].to_dict())
ya = (adult["income"] == ">50K").astype(int)
print("5만 달러 초과 비율:", round(ya.mean(), 3))
print(adult.groupby("sex")["income"].apply(lambda s: (s == ">50K").mean()).round(3).to_string())

48,842명 가운데 23.9%가 5만 달러를 넘습니다. 은행 기록처럼 빈칸은 없지만, 직업 2,809칸과 일하는 곳 2,799칸에 "?"가 적혀 있습니다. 여자는 10.9%, 남자는 30.4%로 차이가 큽니다.

**2단계: 기준선과 나무, 숲**([파이썬 실습 10-3], [10-6] 응용). 무작위로 75%와 25%로 나누어 기준선, 깊이 3 나무, 랜덤 포레스트를 견주시오. 은행 기록과 달리 무작위로 나누어도 되는 까닭을 쓰시오.

In [ ]:
XA = pd.get_dummies(adult.drop(columns=["income"]), dtype=int)
Atr, Ate, atr, ate = train_test_split(XA, ya, test_size=0.25, stratify=ya,
                                      random_state=42)
print("특징", XA.shape[1], "개, 기준선 정확도", round(1 - ate.mean(), 3))
a3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Atr, atr)
print(export_text(a3, feature_names=list(XA.columns), max_depth=1))
arf = RandomForestClassifier(n_estimators=200, random_state=0,
                             n_jobs=-1).fit(Atr, atr)        # 기본 설정 그대로
for name, mdl in [("깊이 3 나무", a3), ("랜덤 포레스트", arf)]:
    print(f"{name}: 정확도 {mdl.score(Ate, ate):.3f}  AUC {auc(mdl, Ate, ate):.3f}")

이 데이터는 한 해에 한 번 조사한 것이고 한 사람이 한 행이라, 무작위로 나누어도 시험 기록의 "옆 날짜"가 훈련에 들어갈 걱정이 없습니다. 기준선은 76.1%이고, 깊이 3 나무는 AUC 0.858, 랜덤 포레스트는 0.902입니다. 나무의 첫 질문은 "배우자와 함께 사는 기혼인가?"입니다.

**3단계: 중요도 의심하기**([파이썬 실습 10-8] 응용). 랜덤 포레스트의 불순도 중요도 상위 변수를 구하고, 그 변수들을 하나씩 섞었을 때 AUC가 얼마나 떨어지는지 구하시오.

In [ ]:
imp_a = pd.Series(arf.feature_importances_, XA.columns)
print("불순도 중요도 상위 4\n", imp_a.sort_values(ascending=False).head(4).round(3))
base = auc(arf, Ate, ate)
rng = np.random.default_rng(0)
for col in ["fnlwgt", "age", "capital_gain", "hours"]:     # 한 열만 섞어 보기
    S = Ate.copy()
    S[col] = rng.permutation(S[col].values)
    print(f"{col:13s} 섞으면 AUC {base:.3f} → {auc(arf, S, ate):.3f}")

불순도 중요도 1위는 fnlwgt(0.166)입니다. 그런데 fnlwgt를 섞어도 AUC는 0.902 그대로입니다. 자본 소득을 섞으면 0.862로 떨어집니다. fnlwgt는 값이 거의 사람마다 달라서, 나무가 질문의 문턱으로 쓸 재료가 많습니다. 그래서 우연한 무늬를 쫓는 질문에 자주 쓰이고, 불순도 중요도가 부풀었습니다. 쉽게 말하면, 불순도 중요도는 "질문에 많이 쓰였다"이지 "판정에 꼭 필요했다"가 아닙니다.

**4단계: 결과 해석과 한계.** 남자와 여자로 나누어, 실제로 5만 달러를 넘는 사람 가운데 모델이 찾은 비율(재현율)을 구하시오. 이 차이와 데이터의 한계를 [표 10-4]처럼 쓰시오.

In [ ]:
pa = arf.predict_proba(Ate)[:, 1] >= 0.5
for s in ["Male", "Female"]:
    k = (adult.loc[Ate.index, "sex"] == s).values
    hit = pa[k] & (ate.values[k] == 1)
    print(f"{s:6s}: 실제 5만 달러 초과 {ate.values[k].sum():4d}명, 모델이 찾은 비율 "
          f"{hit.sum() / ate.values[k].sum():.3f}")

실제로 5만 달러를 넘는 사람 가운데 모델이 찾은 비율은 남자 64.1%, 여자 54.3%입니다. 고소득자가 적은 여자 쪽에서는, 모델이 고소득이라고 판정하는 데 더 인색했던 것으로 보입니다. 9장 6.3의 공정함 이야기가 여기서도 나타납니다. 1994년 미국이라는 시기와 나라, 5만 달러라는 문턱도 한계로 적어야 합니다.

**제출물.** 그림 2장(깊이 3 나무, 두 가지 중요도), 성적표 1개(기준선, 나무, 숲의 정확도와 AUC), 결론 5문장 이내.

**[표 10-5] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | "?" 칸, 중복, fnlwgt의 뜻을 모두 밝힘 | "?" 칸만 셈 | 점검 없이 바로 모델을 만듦 |
| 기준선 비교(20%) | 기준선과 견주고, 무작위 분할이 괜찮은 까닭을 씀 | 기준선은 있으나 나누는 방법을 설명하지 않음 | 기준선 없이 정확도만 씀 |
| 기법 적용(30%) | 나무와 숲의 성적, 두 중요도가 맞음 | 성적은 맞으나 중요도를 하나만 씀 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | fnlwgt를 왜 믿으면 안 되는지 설명하고 성별 차이와 한계를 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |

교수용 정답 예시: 기준선 76.1%, 깊이 3 나무 AUC 0.858, 랜덤 포레스트 AUC 0.902, fnlwgt 불순도 중요도 0.166(섞어도 AUC 변화 없음), 재현율 남자 64.1%, 여자 54.3%.

## ✏️ [연습문제 10-6] 달력 정보를 빼면

3.8의 의심을 확인해 보시오. [파이썬 실습 10-3]의 특징에서 날짜(day)와 달(month_로 시작하는 열)을 모두 빼고 학습률 0.05의 부스팅을 다시 배우게 한 뒤, 시험 AUC를 구해 원래 모델(0.692)과 견주시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오. [파이썬 실습 10-7]의 `boost( )` 함수를 그대로 씁니다.

In [ ]:
drop = [c for c in X.columns if c == "day" or c.startswith("month_")]
gb2 = boost(0.05, Xtr.drop(columns=drop), ytr)
print("뺀 열:", len(drop), "개 / 시험 AUC:", round(auc(gb2, Xte.drop(columns=drop), yte), 3))